# Loopcom Yiddish learning engine — Kaggle free-GPU LABELLING run

Pushed and started by `kaggle-label.ts run`. The attached dataset
(`kernel-metadata.json` -> `dataset_sources`) is `kaggle-label.ts pack`'s
output folder: `manifest.json` (one row per audio file: `itemId`, `assetId`,
`sourceKey`, `file`, `durationMs`) plus `audio/<assetId>.ogg` (16kHz mono
Opus, ~24kbps — small on purpose so a ~15GB batch stays a fast upload).

This notebook transcribes every file in the manifest with
`ivrit-ai/yi-whisper-large-v3-turbo-ct2` (the model production already
serves) on however many GPUs Kaggle gave this session — 2x T4 normally, with
a safe fallback to one worker if only one GPU (or none) shows up — and writes
`/kaggle/working/transcripts.json`. `kaggle-label.ts pull` downloads it back
afterwards, and `kaggle-label.ts import` turns it into `YcTranscript` rows.

No training happens here — this notebook is labelling only.

In [ ]:
import subprocess, sys

# faster-whisper is NOT on the base Kaggle GPU image (unlike torch, which is
# — see requirements-kaggle.txt's comment in ../yiddish-finetune for why that
# file never touches torch). Quiet install; ctranslate2 pulls in a CUDA
# runtime that matches the image's driver.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper"], check=True)

In [ ]:
import glob, json, os

import torch

# Kaggle mounts every dataset in kernel-metadata.json's dataset_sources under
# /kaggle/input/<slug>/. manifest.json sits at that folder's root.
manifest_matches = glob.glob("/kaggle/input/*/manifest.json") or glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert manifest_matches, "no manifest.json found under /kaggle/input — check kernel-metadata.json dataset_sources"
MANIFEST_PATH = manifest_matches[0]
DATASET_DIR = os.path.dirname(MANIFEST_PATH)
print("dataset dir:", DATASET_DIR)

with open(MANIFEST_PATH, encoding="utf-8") as f:
    manifest = json.load(f)
print("manifest entries:", len(manifest))

NUM_GPUS = torch.cuda.device_count()
print("CUDA available:", torch.cuda.is_available(), "GPU count:", NUM_GPUS)

In [ ]:
MODEL_NAME = "ivrit-ai/yi-whisper-large-v3-turbo-ct2"


def transcribe_entries(entries, device_index):
    """Runs in its own process (see ProcessPoolExecutor below) — loads the
    model once per process, on its own GPU, and transcribes its half of the
    manifest. Never raises: a per-file failure is recorded as
    {"error": str(exc)} and the loop continues, so one bad file never costs
    the rest of the batch."""
    from faster_whisper import WhisperModel

    model = WhisperModel(MODEL_NAME, device="cuda", device_index=device_index, compute_type="float16")
    results = {}
    for i, entry in enumerate(entries):
        asset_id = entry["assetId"]
        audio_path = os.path.join(DATASET_DIR, entry["file"])
        try:
            segments_iter, info = model.transcribe(
                audio_path,
                language="yi",
                beam_size=1,
                vad_filter=True,
                word_timestamps=True,
            )
            segments = []
            for seg in segments_iter:
                words = [
                    {"word": w.word, "start": w.start, "end": w.end, "probability": w.probability}
                    for w in (seg.words or [])
                ]
                segments.append(
                    {
                        "start": seg.start,
                        "end": seg.end,
                        "text": seg.text,
                        "avg_logprob": seg.avg_logprob,
                        "no_speech_prob": seg.no_speech_prob,
                        "words": words,
                    }
                )
            results[asset_id] = {"language": info.language, "duration": info.duration, "segments": segments}
        except Exception as exc:  # noqa: BLE001 - one bad file must not sink the batch
            results[asset_id] = {"error": str(exc)}
        if (i + 1) % 10 == 0:
            print(f"[gpu{device_index}] {i + 1}/{len(entries)} done", flush=True)
    return results

In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor

workers = 2 if NUM_GPUS >= 2 else 1
print(f"using {workers} worker process(es) for {len(manifest)} file(s)")

t0 = time.time()
if workers == 2:
    mid = len(manifest) // 2
    halves = [manifest[:mid], manifest[mid:]]
    with ProcessPoolExecutor(max_workers=2) as ex:
        futures = [ex.submit(transcribe_entries, halves[i], i) for i in range(2)]
        all_results = {}
        for fut in futures:
            all_results.update(fut.result())
else:
    all_results = transcribe_entries(manifest, 0)
elapsed = time.time() - t0

n_ok = sum(1 for r in all_results.values() if "error" not in r)
n_err = sum(1 for r in all_results.values() if "error" in r)
print(f"transcribed {len(all_results)} file(s) in {elapsed:.1f}s ({n_ok} ok, {n_err} error(s))")

In [ ]:
out = {"model": MODEL_NAME, "results": all_results}
with open("/kaggle/working/transcripts.json", "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False)

total_hours = sum(r.get("duration", 0) for r in all_results.values() if "error" not in r) / 3600
print(f"wrote transcripts.json: {n_ok} ok, {n_err} error(s), {total_hours:.2f} hours transcribed")